# 03 — Feature engineering & cross-validation

**Variant:** `robust_scale_log1p` · ColumnTransformer + fold indices (Pedregosa 2011 pipeline pattern)


In [ ]:
"""Shared imports for Rogii TVT pipeline notebooks."""
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NB_DIR = Path.cwd()
if not (NB_DIR / "phase_runner.py").is_file():
    NB_DIR = Path(r"/lustre/work/sweeden/agent-tracing-trace-baseline/examples/rogii/traces/preprocessing/robust_scale_log1p/notebooks")
VARIANT_DIR = NB_DIR.parent
ROGII_ROOT = Path("/lustre/work/sweeden/rogii")
sys.path.insert(0, str(NB_DIR))
if str(ROGII_ROOT) not in sys.path:
    sys.path.insert(0, str(ROGII_ROOT))

from phase_runner import (
    ARTIFACTS_ROOT,
    TRACE_INDEX,
    require_prior_phase,
    load_phase_manifest,
    trace_steps_for_phase,
    _resolve_path,
    _read_json,
    _load_train_predict,
)

pd.set_option("display.max_columns", 40)
plt.style.use("seaborn-v0_8-whitegrid")

PHASE = "03_feature_engineering"


## 1. Load phase 01 data paths


In [ ]:
p01 = load_phase_manifest("01_data_analysis")
paths = _read_json(_resolve_path(p01["outputs"]["data_paths"]))
schema = _read_json(_resolve_path(p01["outputs"]["schema"]))
target = p01["target_column"]
id_col = schema["id_column"]
train_df = pd.read_csv(paths["train_csv"])
test_df = pd.read_csv(paths["test_csv"])
print(train_df.shape, test_df.shape)


## 2. Feature column selection


In [ ]:
tp = _load_train_predict()
train_df = tp.align_train_target_to_schema(train_df, target)
feature_cols = [c for c in train_df.columns if c not in (id_col, target) and c in test_df.columns]
print(f"{len(feature_cols)} shared feature columns")
print(feature_cols)


## 3. Numeric / low-card / high-card split


In [ ]:
from pipeline.preprocessor import replace_sentinels_with_nan
X_train = replace_sentinels_with_nan(train_df[feature_cols].copy())
X_test = replace_sentinels_with_nan(test_df[feature_cols].copy())
num, low_c, high_c = tp.categorize_columns(pd.concat([X_train, X_test], ignore_index=True), id_col=id_col, target_cols=[target])
print("numeric:", num)
print("low_card:", low_c)
print("high_card:", high_c)


## 4. ColumnTransformer assembly preview


In [ ]:
preprocessor = tp.build_feature_matrix(X_train, numeric_cols=num, low_card_cols=low_c, high_card_cols=high_c)
print(preprocessor)


## 5. Effective CV scheme


In [ ]:
scheme, groups, n_eff = tp._effective_cv(train_df, test_df, id_col=id_col, n_splits_req=5)
print("scheme:", scheme, "| n_splits:", n_eff, "| groups:", groups is not None)


## 6. Fold index sizes (leak-safe splits)


In [ ]:
from pipeline.cv_orchestrator import emit_fold_indices
fold_sizes = []
for i, (tr, va) in enumerate(emit_fold_indices(
    scheme if scheme == "groupkfold" else "kfold", X_train, n_splits=n_eff,
    groups=groups, shuffle=True, random_state=42,
)):
    fold_sizes.append({"fold": i, "train": len(tr), "val": len(va)})
pd.DataFrame(fold_sizes)


## 7. Visualize fold validation sizes


In [ ]:
fs = pd.DataFrame(fold_sizes)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(fs["fold"].astype(str), fs["val"], label="val", alpha=0.8)
ax.bar(fs["fold"].astype(str), fs["train"], bottom=fs["val"], label="train", alpha=0.5)
ax.set_xlabel("fold"); ax.set_ylabel("rows"); ax.legend(); ax.set_title("CV fold row counts")
plt.tight_layout(); plt.show()


## 8. Depth subdivision preview


In [ ]:
from pipeline.cv_orchestrator import subdivide_train_by_depth_bin
if "MD" in train_df.columns:
  depth_sub = subdivide_train_by_depth_bin(train_df)
  print(json.dumps(depth_sub, indent=2)[:800])


## 9. Numeric feature distributions (sample)

Inspect a few log curves after sentinel replacement — matches EDA sections in Pilkwang's leakage-aware pipeline.


In [ ]:
plot_cols = [c for c in num[:4] if c in X_train.columns]
fig, axes = plt.subplots(1, len(plot_cols), figsize=(3 * len(plot_cols), 3))
if len(plot_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, plot_cols):
    ax.hist(X_train[col].dropna(), bins=30, color="steelblue", alpha=0.85)
    ax.set_title(col)
plt.tight_layout(); plt.show()


## 10. Categorical cardinality check


In [ ]:
for col in (low_c + high_c)[:6]:
    print(col, "nunique:", X_train[col].nunique(dropna=True))


## 11. Train/test row alignment


In [ ]:
print("train rows:", len(train_df), "test rows:", len(test_df))
print("shared features:", len(feature_cols))


## 12. Missing rate after sentinel replacement


In [ ]:
print(X_train.isna().mean().sort_values(ascending=False).head(10))


## 13. Feature matrix shape after transform (sample)


In [ ]:
Xt = preprocessor.fit_transform(X_train.iloc[:500])
print("transformed shape (500 rows):", getattr(Xt, "shape", None))


## 14. Group overlap train/test


In [ ]:
from pipeline import well_group_detector as wgd
gk = wgd.recommend_group_key(train_df, test_df, id_column=id_col)
print("group_key:", gk)


## 15. Persist phase 03 artifacts


In [ ]:
from phase_runner import run_03_feature_engineering
manifest = run_03_feature_engineering()
print(json.dumps(manifest, indent=2))


## 16. Feature config snapshot


In [ ]:
feat = _read_json(_resolve_path(manifest["outputs"]["feature_config"]))
print(json.dumps(feat, indent=2)[:1200])


## 17. CV config snapshot


In [ ]:
cv_cfg = _read_json(_resolve_path(manifest["outputs"]["cv_config"]))
print(json.dumps(cv_cfg, indent=2))


## 18. Trace alignment check


In [ ]:
print("trace steps:", len(trace_steps_for_phase(PHASE)))


## 19. Fold index artifact preview


In [ ]:
fold_path = ARTIFACTS_ROOT / PHASE / "fold_indices.json"
if fold_path.is_file():
    fi = _read_json(fold_path)
    print("n_folds:", len(fi.get("folds", fi)))


## 20. Handoff to model training

Phase 04 loads `feature_config`, `cv_config`, and fold indices for LightGBM CV.
